### Importation des bibliothèques

### Installation des d?pendances (Kaggle-safe)


In [ ]:
import sys
import subprocess
import importlib.util


def ensure_package(import_name: str, pip_spec: str):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_spec}")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            pip_spec,
        ])


# Kaggle-safe: do not force-upgrade torch/cuda.
ensure_package("numpy", "numpy")
ensure_package("scipy", "scipy")
ensure_package("pandas", "pandas")
ensure_package("sklearn", "scikit-learn")
ensure_package("yaml", "pyyaml")
ensure_package("gensim", "gensim")
ensure_package("tqdm", "tqdm")
ensure_package("torch", "torch")

print("Dependencies check done")


In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse as sp
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm


### Configuration des chemins et des dossiers de travail

In [ ]:
def detect_project_root():
    candidates = [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path.cwd().resolve() / "Projet_PPD",
        Path.cwd().resolve().parent / "Projet_PPD",
    ]
    for candidate in candidates:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Impossible de trouver le dossier projet contenant data/ et output/.")


BASE_DIR = detect_project_root()
DATA_ROOT = BASE_DIR / "data"
KMEANS_ROOT = BASE_DIR / "output" / "KMeans"
KMEANS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Data: {DATA_ROOT}")
print(f"KMeans output: {KMEANS_ROOT}")


### Initialisation du seed et chargement des datasets

In [ ]:
SEED = 42


def set_seed(seed):
    import random

    random.seed(seed)
    np.random.seed(seed)


def load_dataset(name):
    path = DATA_ROOT / name
    train = sp.load_npz(path / "train_bow.npz").toarray()
    test = sp.load_npz(path / "test_bow.npz").toarray()
    with open(path / "vocab.txt", encoding="utf-8") as f:
        vocab = [w.strip() for w in f if w.strip()]
    labels_train = np.loadtxt(path / "train_labels.txt", dtype=int)
    labels_test = np.loadtxt(path / "test_labels.txt", dtype=int)
    return train, test, vocab, labels_train, labels_test


GENSIM_CACHE = {}


def get_gensim_artifacts(dataset, max_docs=10000):
    if dataset in GENSIM_CACHE:
        return GENSIM_CACHE[dataset]

    train_bow, _, vocab, _, _ = load_dataset(dataset)
    texts = []
    for doc in train_bow[:max_docs]:
        idx = np.where(doc > 0)[0]
        words = [vocab[i] for i in idx if i < len(vocab)]
        if len(words) >= 2:
            texts.append(words)

    dictionary = Dictionary(texts)
    GENSIM_CACHE[dataset] = (texts, dictionary)
    return GENSIM_CACHE[dataset]


set_seed(SEED)
print("Fonctions pr?tes")


### Fonctions pour les métriques d'évaluation

In [ ]:
def topic_diversity(topics):
    words = [w for t in topics for w in t]
    return len(set(words)) / len(words) if words else 0.0


def purity_score(y_true, y_pred):
    total = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        labels, counts = np.unique(y_true[idx], return_counts=True)
        total += counts.max()
    return total / max(len(y_true), 1)


def compute_nmi(theta, labels):
    pred = theta.argmax(axis=1)
    n = min(len(pred), len(labels))
    return normalized_mutual_info_score(labels[:n], pred[:n])


def compute_cv_gensim(beta, vocab, dataset, topn=15):
    texts, dictionary = get_gensim_artifacts(dataset)

    topics = []
    for k in range(beta.shape[0]):
        idx = np.argsort(beta[k])[::-1][:topn]
        topic_words = [
            vocab[i]
            for i in idx
            if i < len(vocab) and vocab[i] in dictionary.token2id
        ]
        if len(topic_words) >= 2:
            topics.append(topic_words)

    if not topics:
        return 0.0

    cm = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v",
    )
    return float(cm.get_coherence())


### Calcul des métriques d'évaluation pour K-Means

In [ ]:
def compute_metrics_kmeans(dataset, K, seed=42, topn=15):
    dataset_dir = KMEANS_ROOT / dataset
    mat_path = dataset_dir / f"{dataset}_KMeans_K{K}_seed{seed}.mat"

    if not mat_path.is_file():
        print(f".mat manquant: {mat_path}")
        return None

    loaded = scipy.io.loadmat(str(mat_path))
    beta = np.asarray(loaded["beta"])
    if beta.shape[0] != K and beta.shape[1] == K:
        beta = beta.T

    train_theta = np.asarray(loaded["train_theta"])
    _, _, vocab, labels_train, _ = load_dataset(dataset)

    topics = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        topics.append([vocab[i] for i in top_idx if i < len(vocab)])
    td = topic_diversity(topics)

    y_pred = train_theta.argmax(axis=1)
    n = min(len(labels_train), len(y_pred))
    pur = purity_score(labels_train[:n], y_pred[:n])
    nmi = compute_nmi(train_theta, labels_train)

    cv = compute_cv_gensim(beta, vocab, dataset, topn=topn)

    res = {
        "Dataset": dataset,
        "Model": "KMeans",
        "K": int(K),
        "Seed": int(seed),
        "C_V": round(cv, 4),
        "TD": round(float(td), 4),
        "Purity": round(float(pur), 4),
        "NMI": round(float(nmi), 4),
    }

    out_path = dataset_dir / f"metrics_{dataset}_KMeans_K{K}_seed{seed}.csv"
    pd.DataFrame([res]).to_csv(out_path, index=False)

    print(
        f"Metrics OK | {dataset} K={K}: "
        f"C_V={res['C_V']:.4f} | TD={res['TD']:.4f} | "
        f"Purity={res['Purity']:.4f} | NMI={res['NMI']:.4f}"
    )
    return res


### Calcul de la coh?rence C_V avec Gensim


In [ ]:
def compute_cv_kmeans(dataset, K, seed=42, topn=15):
    dataset_dir = KMEANS_ROOT / dataset
    mat_path = dataset_dir / f"{dataset}_KMeans_K{K}_seed{seed}.mat"

    if not mat_path.is_file():
        print(f".mat manquant: {mat_path}")
        return None

    loaded = scipy.io.loadmat(str(mat_path))
    beta = np.asarray(loaded["beta"])
    if beta.shape[0] != K and beta.shape[1] == K:
        beta = beta.T

    _, _, vocab, _, _ = load_dataset(dataset)
    cv = compute_cv_gensim(beta, vocab, dataset, topn=topn)
    print(f"C_V (Gensim) {dataset} K={K}: {cv:.4f}")
    return cv


### Entraînement du modèle K-Means 

In [ ]:
def train_one_kmeans(dataset, K, seed=42):
    set_seed(seed)
    total_start = time.perf_counter()

    dataset_dir = KMEANS_ROOT / dataset
    dataset_dir.mkdir(parents=True, exist_ok=True)
    mat_path = dataset_dir / f"{dataset}_KMeans_K{K}_seed{seed}.mat"

    if mat_path.exists():
        return {'mat_path': str(mat_path), 'timing': None}

    print(f"KMeans | {dataset} | K={K}")
    train_bow, test_bow, _, _, _ = load_dataset(dataset)

    train_bow_norm = normalize(train_bow)
    test_bow_norm = normalize(test_bow)

    kmeans = KMeans(
        n_clusters=K,
        init="k-means++",
        random_state=seed,
        n_init=20,
        max_iter=100,
        tol=1e-4,
    )

    train_start = time.perf_counter()
    train_clusters = kmeans.fit_predict(train_bow_norm)
    train_seconds = time.perf_counter() - train_start

    test_clusters = kmeans.predict(test_bow_norm)

    beta = kmeans.cluster_centers_
    beta = beta / (beta.sum(axis=1, keepdims=True) + 1e-10)

    train_theta = np.zeros((len(train_clusters), K), dtype=np.float32)
    train_theta[np.arange(len(train_clusters)), train_clusters] = 1.0
    test_theta = np.zeros((len(test_clusters), K), dtype=np.float32)
    test_theta[np.arange(len(test_clusters)), test_clusters] = 1.0

    scipy.io.savemat(
        str(mat_path),
        {
            "beta": beta.astype(np.float32),
            "train_theta": train_theta,
            "test_theta": test_theta,
        },
    )

    total_seconds = time.perf_counter() - total_start
    timing_row = {
        'model': 'KMeans', 'dataset': dataset, 'K': int(K), 'seed': int(seed), 'device': 'cpu',
        'phase': 'train', 'train_seconds': float(train_seconds), 'total_seconds': float(total_seconds),
        'train_minutes': float(train_seconds / 60.0), 'total_minutes': float(total_seconds / 60.0),
    }

    timing_path = dataset_dir / f"{dataset}_KMeans_K{K}_seed{seed}_timing.csv"
    pd.DataFrame([timing_row]).to_csv(timing_path, index=False)

    print(f"Sauvegard?: {mat_path}")
    print(f"Timing: {timing_path} | train={train_seconds:.2f}s total={total_seconds:.2f}s")

    return {'mat_path': str(mat_path), 'timing': timing_row, 'timing_path': str(timing_path)}


### Batch d'entraînement K-Means sur 4 datasets

In [ ]:
print("=" * 60)
print("ENTRAINEMENT KMEANS - 4 BASES")
print("=" * 60)

DATASETS = ["20NG", "YahooAnswer", "AGNews", "IMDB"]
K_VALUES = [20, 50, 100]

total_processed = 0
total_skipped = 0
total_models = 0
training_time_rows = []

for dataset in DATASETS:
    print(f"\n{'='*50}")
    print(f"BASE: {dataset}")

    for K in K_VALUES:
        total_models += 1
        dataset_dir = KMEANS_ROOT / dataset
        mat_path = dataset_dir / f"{dataset}_KMeans_K{K}_seed{SEED}.mat"

        print(f"\n--- {dataset} K={K} ---")

        if mat_path.exists():
            print(f"SKIP: {mat_path.name}")
            total_skipped += 1
            continue

        print("Calcul KMeans...")
        out = train_one_kmeans(dataset, K, seed=SEED)
        if isinstance(out, dict) and out.get('timing') is not None:
            training_time_rows.append(out['timing'])

        total_processed += 1
        print(f"OK {dataset} K={K}")

print(f"\n{'='*60}")
print(f"Trait?es: {total_processed} | Saut?es: {total_skipped} | Total: {total_models}")

if training_time_rows:
    df_train_times = pd.DataFrame(training_time_rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df_train_times[['dataset', 'K', 'device', 'train_seconds', 'total_seconds']])

    time_csv = KMEANS_ROOT / 'KMeans_training_times.csv'
    df_train_times.to_csv(time_csv, index=False)
    print(f"Saved training times: {time_csv}")


### Calcul des métriques K-Means (C_V + TD + Purity + NMI)


In [ ]:
def read_cv_metrics(dataset, K, seed=42):
    metrics_path = KMEANS_ROOT / dataset / f"metrics_{dataset}_KMeans_K{K}_seed{seed}.csv"
    if not metrics_path.exists():
        print("C_V = NOT FOUND")
        return None

    row = pd.read_csv(metrics_path).iloc[0]
    cv = float(row["C_V"])
    print(f"C_V (Gensim) = {cv:.4f}")
    return cv


In [ ]:
print("=" * 60)
print("METRICS KMEANS (C_V GENSIM + TD + PURITY + NMI)")
print("=" * 60)

all_results = []

for dataset in DATASETS:
    print(f"\n{'='*50}")
    print(f"BASE: {dataset}")

    for K in K_VALUES:
        print(f"\n--- {dataset} K={K} ---")
        res = compute_metrics_kmeans(dataset, K, seed=SEED)
        if res is not None:
            all_results.append(res)
            read_cv_metrics(dataset, K, seed=SEED)

if all_results:
    df_kmeans = pd.DataFrame(all_results).sort_values(["Dataset", "K"])
    display(df_kmeans)
    global_csv = KMEANS_ROOT / "KMeans_ALL_METRICS.csv"
    df_kmeans.to_csv(global_csv, index=False)
    print(f"R?sum? sauvegard?: {global_csv}")


### Analyse comparative : Résultats originaux vs Reproduction

In [14]:
data_papier = {
    'Dataset': ['20NG', '20NG', 'YahooAnswer', 'YahooAnswer', 'AGNews', 'AGNews', 'IMDB', 'IMDB'],
    'K': [50, 100, 50, 100, 50, 100, 50, 100],
    'CV_Papier': [0.251, 0.294, 0.213, 0.244, 0.271,0.297, 0.241, 0.289],
    'TD_Papier': [0.204, 0.317, 0.219, 0.302, 0.242, 0.345, 0.264, 0.395]
}


data_repro = {
    'Dataset': ['20NG', '20NG', 'YahooAnswer', 'YahooAnswer', 'AGNews', 'AGNews', 'IMDB', 'IMDB'],
    'K': [50, 100, 50, 100, 50, 100, 50, 100],
    'CV_Repro': [0.388, 0.389, 0.329, 0.326, 0.389, 0.393, 0.336, 0.334],
    'TD_Repro': [0.316, 0.297, 0.305, 0.279, 0.500, 0.453, 0.105, 0.120],
    'Purity_R': [0.378, 0.401, 0.284, 0.322, 0.573, 0.648, 0.614, 0.624],
    'NMI_R':    [0.310, 0.316, 0.100, 0.120, 0.177, 0.192, 0.023, 0.023]
}

df_p = pd.DataFrame(data_papier)
df_r = pd.DataFrame(data_repro)
df_final = pd.merge(df_p, df_r, on=['Dataset', 'K'])

# Calcul des écarts (Repro - Papier)
df_final['Diff_CV'] = (df_final['CV_Repro'] - df_final['CV_Papier']).round(3)
df_final['Diff_TD'] = (df_final['TD_Repro'] - df_final['TD_Papier']).round(3)


colonnes = [
    'Dataset', 'K', 
    'CV_Repro', 'TD_Repro', 'Purity_R', 'NMI_R', # REPRODUCTION
    'CV_Papier', 'TD_Papier',                    # PAPIER
    'Diff_CV', 'Diff_TD'                         # ECARTS
]

print("="*125)
print(f"{'VOTRE REPRODUCTION (M2)':>55} | {'PAPIER ORIGINAL':>22} | {'ECARTS':>18}")
print("="*125)
print(df_final[colonnes].to_string(index=False))

                                VOTRE REPRODUCTION (M2) |        PAPIER ORIGINAL |             ECARTS
    Dataset   K  CV_Repro  TD_Repro  Purity_R  NMI_R  CV_Papier  TD_Papier  Diff_CV  Diff_TD
       20NG  50     0.388     0.316     0.378  0.310      0.251      0.204    0.137    0.112
       20NG 100     0.389     0.297     0.401  0.316      0.294      0.317    0.095   -0.020
YahooAnswer  50     0.329     0.305     0.284  0.100      0.213      0.219    0.116    0.086
YahooAnswer 100     0.326     0.279     0.322  0.120      0.244      0.302    0.082   -0.023
     AGNews  50     0.389     0.500     0.573  0.177      0.271      0.242    0.118    0.258
     AGNews 100     0.393     0.453     0.648  0.192      0.297      0.345    0.096    0.108
       IMDB  50     0.336     0.105     0.614  0.023      0.241      0.264    0.095   -0.159
       IMDB 100     0.334     0.120     0.624  0.023      0.289      0.395    0.045   -0.275
